# SHAP Explainability


In [1]:
#1

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

# Paths relative to the project root
EXPORT_DIR = Path("model_exports")
MODEL_PATH = EXPORT_DIR / "mlp_model.keras"
SCALER_PATH = EXPORT_DIR / "input_scaler.pkl"
METADATA_PATH = EXPORT_DIR / "model_metadata.json"
DATA_PATH = Path("chiller_predictive_maintenance_dataset.csv")

# Confirm all files exist before loading
for path in [MODEL_PATH, SCALER_PATH, METADATA_PATH, DATA_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path.resolve()}")

# Load metadata, model, and fitted scaler
with METADATA_PATH.open("r", encoding="utf-8") as f:
    metadata = json.load(f)

mlp_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
input_scaler = joblib.load(SCALER_PATH)

input_cols = metadata["input_columns"]
output_cols = metadata["output_columns"]

# Check the metadata dimensions
assert len(input_cols) == 17, f"Expected 17 inputs, got {len(input_cols)}"
assert len(output_cols) == 26, f"Expected 26 outputs, got {len(output_cols)}"
assert len(set(input_cols)) == len(input_cols), "Input column names contain duplicates."
assert len(set(output_cols)) == len(output_cols), "Output column names contain duplicates."

print(f"Inputs: {len(input_cols)}")
print(f"Outputs: {len(output_cols)}")
print(f"Model input shape: {mlp_model.input_shape}")
print(f"Model output shape: {mlp_model.output_shape}")
print(f"Scaler expects features: {input_scaler.n_features_in_}")

Inputs: 17
Outputs: 26
Model input shape: (None, 17)
Model output shape: (None, 26)
Scaler expects features: 17


In [2]:
#2
import pandas as pd
import numpy as np

DATA_PATH = "chiller_predictive_maintenance_dataset.csv"

data = pd.read_csv(DATA_PATH)

# Use only training rows; holdout_manual is not used as SHAP background data
train_df = data.loc[data["Split"] == "train"].copy()
X_train = train_df.loc[:, input_cols]

# Apply the already-fitted scaler. Do not fit it again.
X_train_scaled = input_scaler.transform(X_train)

# Select one output by its exact metadata name
TARGET_OUTPUT = "Chiller_Condenser_Fault"
target_index = output_cols.index(TARGET_OUTPUT)

def predict_selected_output(X_scaled):
    predictions = mlp_model.predict(
        np.asarray(X_scaled, dtype=np.float32),
        verbose=0
    )
    return predictions[:, target_index]

print("Selected output:", TARGET_OUTPUT)
print("Output index:", target_index)
print("Sample prediction:", predict_selected_output(X_train_scaled[:1])[0])

Selected output: Chiller_Condenser_Fault
Output index: 0
Sample prediction: 0.4657815


In [3]:
#3
rng = np.random.default_rng(42)

n_background = min(100, len(X_train_scaled))
background_idx = rng.choice(
    len(X_train_scaled),
    size=n_background,
    replace=False
)
background_scaled = X_train_scaled[background_idx]

print("Background rows:", len(background_scaled))

Background rows: 100


In [4]:
# Sample 500 training rows, excluding the SHAP background rows
available_idx = np.setdiff1d(
    np.arange(len(X_train_scaled)),
    background_idx
)

n_explain = min(500, len(available_idx))
explain_idx_all = rng.choice(
    available_idx,
    size=n_explain,
    replace=False
)

X_explain_all = X_train_scaled[explain_idx_all]

print("Training rows used for explanations:", len(X_explain_all))
print("Background rows:", len(background_scaled))

Training rows used for explanations: 500
Background rows: 100


In [5]:
#5
import shap

# Each output gets its own scalar prediction function and explainer.
def make_output_predictor(output_index):
    def predict_output(X_scaled):
        predictions = mlp_model.predict(
            np.asarray(X_scaled, dtype=np.float32),
            verbose=0
        )
        return predictions[:, output_index]
    return predict_output


shap_values_by_output = {}
base_values_by_output = {}

max_evals = 2 * len(input_cols) + 1  # 35 for 17 inputs

for output_index, output_name in enumerate(output_cols):
    print(f"Explaining {output_index + 1}/26: {output_name}")

    output_predictor = make_output_predictor(output_index)

    output_explainer = shap.PermutationExplainer(
        output_predictor,
        background_scaled
    )

    output_explanation = output_explainer(
    X_explain_all,  # changed from X_explain_small
    max_evals=max_evals
    )

    values = np.asarray(output_explanation.values)

    expected_shape = (len(X_explain_all), len(input_cols))  # changed here

    if values.shape != expected_shape:
        raise ValueError(
            f"{output_name}: expected {expected_shape}, got {values.shape}"
        )

    shap_values_by_output[output_name] = values
    base_values_by_output[output_name] = np.asarray(
        output_explanation.base_values
    )

print(f"Finished: {len(shap_values_by_output)} outputs explained.")

Explaining 1/26: Chiller_Condenser_Fault


PermutationExplainer explainer: 501it [06:15,  1.33it/s]                         


Explaining 2/26: Chiller_ExpansionValve_Fault


PermutationExplainer explainer: 501it [06:00,  1.35it/s]                         


Explaining 3/26: Chiller_Evaporator_Fault


...

Explaining 25/26: General_LowPerformance


PermutationExplainer explainer: 501it [02:37,  2.99it/s]                         


Explaining 26/26: General_UnitUndersizedOversized


PermutationExplainer explainer: 501it [02:40,  2.92it/s]                         

Finished: 26 outputs explained.


In [6]:
#6
importance_df = pd.DataFrame(
    {
        output_name: np.abs(shap_values_by_output[output_name]).mean(axis=0)
        for output_name in output_cols
    },
    index=input_cols
).T

importance_df.index.name = "output"
importance_df.columns.name = "input_feature"

assert importance_df.shape == (26, 17)
assert np.isfinite(importance_df.to_numpy()).all()

# Retain the global importance results in the current notebook folder
output_path = "shap_global_importance.csv"
importance_df.to_csv(output_path)

display(importance_df)
print(f"Saved {importance_df.shape[0]} outputs × {importance_df.shape[1]} inputs to {output_path}")

input_feature,F36,F15,H25,H24,H64,H15,F14,H42,Mode_1Cool_2Heat,H23F,H23C,P02,H20,P03,F37,F39,H62
output,,,,,,,,,,,,,,,,,
Chiller_Condenser_Fault,0.001841,0.003712,0.091899,0.001605,0.035179,0.002787,0.000719,0.001320,0.000874,0.000967,0.000785,0.001078,0.000924,0.001490,0.002448,0.002141,0.006649
Chiller_ExpansionValve_Fault,0.001182,0.001913,0.002011,0.001628,0.001296,0.001297,0.000866,0.086364,0.000807,0.000957,0.000778,0.000928,0.000454,0.002231,0.003435,0.003108,0.003745
Chiller_Evaporator_Fault,0.001625,0.000858,0.007419,0.004714,0.007133,0.004666,0.000544,0.004956,0.002261,0.002696,0.002880,0.017379,0.001519,0.021826,0.016819,0.019079,0.045897
Chiller_Compressor_Fault,0.001585,0.000605,0.003715,0.020881,0.009203,0.021638,0.000397,0.011732,0.006920,0.009079,0.009783,0.003107,0.000464,0.003668,0.005047,0.005579,0.010982
Chiller_General_Fault,0.013481,0.001679,0.009373,0.002262,0.038717,0.003141,0.001024,0.001795,0.001071,0.001239,0.001343,0.002311,0.000458,0.002646,0.026301,0.048300,0.005512
Condenser_Valve_Fault,0.002771,0.006098,0.138135,0.001084,0.028155,0.001659,0.001301,0.001159,0.001480,0.000915,0.000729,0.001351,0.002067,0.001920,0.002092,0.002308,0.005768
Condenser_Filter_Fault,0.002646,0.006170,0.138155,0.001029,0.028358,0.001618,0.001305,0.001174,0.001468,0.000924,0.000768,0.001282,0.002083,0.001926,0.001904,0.002240,0.005634
Condenser_GasChargeNeeded,0.002819,0.004523,0.226064,0.001922,0.030522,0.002138,0.000881,0.001664,0.001372,0.001059,0.000813,0.002027,0.000794,0.002142,0.002833,0.002781,0.003967
Condenser_Fan_Fault,0.004628,0.015893,0.086335,0.002702,0.095882,0.003101,0.001186,0.002491,0.002979,0.001485,0.001761,0.001867,0.002218,0.003808,0.003766,0.004261,0.007089


Saved 26 outputs × 17 inputs to shap_global_importance.csv


In [7]:
predictions_all = mlp_model.predict(X_explain_all, verbose=0)

for output_index, output_name in enumerate(output_cols):
    base = np.asarray(base_values_by_output[output_name]).reshape(-1)

    if base.size == 1:
        base = np.full(len(X_explain_all), base.item())

    reconstructed = (
        base + shap_values_by_output[output_name].sum(axis=1)
    )
    errors = np.abs(reconstructed - predictions_all[:, output_index])

    print(
        f"{output_name}: "
        f"mean abs error={errors.mean():.6g}, "
        f"max abs error={errors.max():.6g}"
    )

Chiller_Condenser_Fault: mean abs error=0, max abs error=0
Chiller_ExpansionValve_Fault: mean abs error=1.44329e-18, max abs error=1.11022e-16
Chiller_Evaporator_Fault: mean abs error=1.11022e-19, max abs error=5.55112e-17
Chiller_Compressor_Fault: mean abs error=0, max abs error=0
Chiller_General_Fault: mean abs error=1.66533e-19, max abs error=2.77556e-17
Condenser_Valve_Fault: mean abs error=8.63198e-18, max abs error=1.11022e-16
Condenser_Filter_Fault: mean abs error=1.11022e-17, max abs error=1.11022e-16
Condenser_GasChargeNeeded: mean abs error=1.08524e-17, max abs error=2.22045e-16
Condenser_Fan_Fault: mean abs error=7.68829e-18, max abs error=1.11022e-16
Condenser_Coil_Fault: mean abs error=9.32587e-18, max abs error=1.11022e-16
Condenser_GasLeak: mean abs error=9.13158e-18, max abs error=1.11022e-16
Evaporator_LiquidLeak: mean abs error=2.08167e-18, max abs error=1.11022e-16
Evaporator_Valve_Fault: mean abs error=3.13638e-18, max abs error=1.11022e-16
Evaporator_StrainerFilter

In [8]:
shap_archive = {
    "input_columns": np.array(input_cols, dtype=str),
    "output_columns": np.array(output_cols, dtype=str),
    "background_indices": background_idx,
    "explanation_indices": explain_idx_all,
}

for output_name in output_cols:
    shap_archive[f"shap_values__{output_name}"] = (
        shap_values_by_output[output_name]
    )
    shap_archive[f"base_values__{output_name}"] = (
        base_values_by_output[output_name]
    )

np.savez_compressed("shap_results_500_rows.npz", **shap_archive)

In [9]:
def rank_fault_outputs(predictions, top_n=10):
    scores = np.asarray(predictions, dtype=float)

    # Accept either one prediction vector or a one-row model result
    if scores.ndim == 2 and scores.shape[0] == 1:
        scores = scores[0]

    if scores.ndim != 1 or len(scores) != len(output_cols):
        raise ValueError(
            f"Expected one score for each of {len(output_cols)} outputs; "
            f"got shape {scores.shape}"
        )

    if not np.isfinite(scores).all():
        raise ValueError("Predictions contain non-finite values.")

    ranked = pd.DataFrame({
        "output": output_cols,
        "model_score": scores
    })

    ranked = ranked.sort_values(
        "model_score",
        ascending=False,
        kind="stable"
    ).reset_index(drop=True)

    ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))

    return ranked.head(top_n)


def predict_and_rank(sensor_measurements, top_n=10):
    missing = [col for col in input_cols if col not in sensor_measurements]
    if missing:
        raise ValueError(f"Missing input measurements: {missing}")

    # Apply the existing fitted scaler in the metadata's feature order
    input_row = pd.DataFrame(
        [[sensor_measurements[col] for col in input_cols]],
        columns=input_cols
    )
    input_scaled = input_scaler.transform(input_row)

    predictions = mlp_model.predict(input_scaled, verbose=0)
    return rank_fault_outputs(predictions, top_n=top_n)


# Demonstrate with one existing training row
example_measurements = X_train.iloc[0].to_dict()
display(predict_and_rank(example_measurements, top_n=10))

,rank,output,model_score
0,1,Compressor_General_Fault,0.806502
1,2,Evaporator_StrainerFilter_Fault,0.631548
2,3,Chiller_Compressor_Fault,0.498490
3,4,Condenser_Coil_Fault,0.490119
4,5,Condenser_GasLeak,0.490119
5,6,Condenser_Fan_Fault,0.489154
6,7,Chiller_Evaporator_Fault,0.478613
7,8,Chiller_Condenser_Fault,0.465782
8,9,Condenser_GasChargeNeeded,0.292885
9,10,Chiller_General_Fault,0.272692
